# Figure 2d — TetraSpeck bead precision, by excitation wavelength

**Makes up:** Figure 2d (position precision $\sigma_{xy}$ and colour precision $\sigma_\mathrm{colour}$
of single TetraSpeck beads against signal, with $a/\sqrt{N}+c$ fits, all three lasers pooled as
published). Also, for the reviewers: the same split by excitation laser (488, 561, 638 nm), with
per-laser fits and a comparison of the lasers where their signal ranges overlap.

**Data:** BioStudies `S-BIAD3210`, `2026_Multicolour_Paper/Ximea/TetraspeckBeads/Tetraspeck_<laser>_excitation/<power>_excitation/`:
0.1 µm TetraSpeck beads, 3 lasers × 7 powers (1–10 mW), 25 positions per acquisition (16 for
488 nm / 2 mW), each one TIFF of 200 frames (~430 MB) plus its MicroManager metadata. A stray
`1mW%20excitation` folder is not used.

**Method:** fit each position (`fit_SM_data`, 12 px ROI, PSF wavelength = the emission peak of the
TetraSpeck dye the laser excites) → AIM drift correction (10-frame segments, 20 nm intersection
distance, 60 nm search radius, as in the Methods) → HDBSCAN clustering of each acquisition's
localisations into beads (≥ 10 localisations) → per bead (seen in ≥ 50 of 200 frames), as in the SI
note *Calibrating Precision*:

* $\sigma_{xy}$ = mean of the standard deviations of x and y about the bead's own mean position;
* $\sigma_\mathrm{colour}$ = std / mean of the bead's relative pixel signal $A_X$ on pixel X (1, 2, 3); the
  pixel-2 value is scaled by $\sqrt{2}$, since the Bayer mosaic has twice as many pixel-2 pixels;
* signal $N$ = the bead's mean fitted signal per frame, $A_1+A_2+A_3$ before normalisation.
  For colour, $N_X = N\,\overline{A_X}$, the signal on pixel X.

**Signal units:** $N$ is in **detected photoelectrons per frame**. The fit works on images
converted with the per-pixel gain, offset and relative QE maps in `Camera_Calibrations/Ximea_Camera`.
$N$ is the integrated amplitude of the fitted PSF, background excluded. It is not corrected for the
camera's absolute QE, so it is not the number of photons reaching the sensor.

Quality cuts follow the Methods (χ²ν < 3, localisation error < 1 px, > 100 photoelectrons), with two
exceptions:
1. No cut on the colour error, so frames are not selected on the quantity being measured.
2. A PSF σ window of 75–250 nm instead of the single-dye 75–160 nm. The beads' fitted σ is
   ~145 nm at 561 nm and ~170 nm at 638 nm, so the single-dye window drops 40 % and 90 % of
   their frames.

The published plot's cleaning is kept: σ_xy 0.01–20 nm, σ_colour 0.5–50 %, N < 2 × 10⁴.
Shared code: `_multifov.py` (download / fit / drift / cluster) and `_beads.py` (settings, per-bead
precision, fits).

## FAST_MODE

`FAST_MODE = True` (default) uses one position (`Pos000_000`) per laser and power: 21 TIFFs,
~9 GB. Fitting takes ~2 min per position. The download from EBI usually takes far longer. Every
panel is made, from ~1/25 of the beads (a few to ~35 per acquisition), so the per-laser fits are
loose. `FAST_MODE = False` uses all 516 positions (~220 GB).

---
## 0 — Imports

In [ ]:
import sys
import types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# _biostudies_download.py, _multifov.py and _beads.py live beside this notebook.
sys.path.insert(0, str(Path.cwd().resolve()))
import _multifov as mf
import _beads as bd

from pyS3M import IOFunctions
from pyS3M import sCMOSFunctions
from pyS3M import SR_Functions
from pyS3M import SM_extractionfunctions
from pyS3M import PlottingBase
from pyS3M.DriftCorrectionFunctions import Drift_Correction_Functions

IO = IOFunctions.IO_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
SupRes_F = SR_Functions.SuperRes_Functions()   # Ximea defaults (69 nm pixels)
SM_E = SM_extractionfunctions.extract_SMs()
DC = Drift_Correction_Functions()
plotter = PlottingBase.PublicationPlotter(dark_background=False)

print("Python:", sys.version)

---
## 1 — Configuration

In [ ]:
FAST_MODE = True   # True: one position per laser and power (~9 GB). False: all 516 positions (~220 GB).
REFIT = False      # True: fit again even if fits with the same files and settings exist.
FAST_POSITION = "Pos000_000"
N_BOOT = 1000      # bootstrap resamples (over beads) for the 95 % intervals

REPO_ROOT = Path.cwd().resolve().parents[1]   # notebooks/figures -> repo root
CALIB_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT  = Path("/media/u2473660/JSBLeeLabBackup/Data/20261026_Tetraspeck")   # git-ignored (bare outputs/ rule)
FIG_DIR   = OUT_ROOT / ("fast" if FAST_MODE else "full")
# Point DATA_ROOT at a drive with room for the download. Each acquisition gets its own sub-folder,
# separately for each FAST_MODE setting: the fit reads every TIFF inside it.
DATA_ROOT = OUT_ROOT / ("data_fast" if FAST_MODE else "data_full")
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

calib = dict(
    gain_map=IO.read_tiff(CALIB_DIR / "gain.tif"),
    offset_map=IO.read_tiff(CALIB_DIR / "offset.tif"),
    rqe=IO.read_tiff(CALIB_DIR / "rqe.tif"),
    read_noise=IO.read_tiff(CALIB_DIR / "readnoise.tif"),
    variance=IO.read_tiff(CALIB_DIR / "variance.tif"),
)

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

ACQUISITIONS = [(laser, power) for laser in bd.LASERS for power in bd.POWERS]

print(f"FAST_MODE  : {FAST_MODE}")
print(f"Data dir   : {DATA_ROOT.resolve()}")
print(f"Figures dir: {FIG_DIR.resolve()}")

---
## 2 — Download, fit and drift-correct every acquisition

Per laser and power: download from BioStudies (with each position's metadata, which holds the camera
ROI the fit needs), fit each position, then AIM-correct its drift. Re-running is safe: complete
downloads are skipped, and fits are reused unless the files or settings change (or `REFIT = True`).

In [ ]:
loc_files = {}
for laser, power in ACQUISITIONS:
    fit_params = dict(bd.FIT_PARAMS, peak_wavelength=bd.PEAK_UM[laser])
    print(f"\n=== {laser} nm, {power} (PSF wavelength {fit_params['peak_wavelength']} um)")
    loc_files[laser, power] = mf.prepare_acquisition(
        bd.remote_folder(laser, power), DATA_ROOT / laser / power, FAST_MODE, SupRes_F, calib,
        smoothing_function, IO, fit_params, drift_corrector=DC, drift_params=bd.DRIFT_PARAMS,
        position=FAST_POSITION if FAST_MODE else None, refit=REFIT,
    )

for (laser, power), files in loc_files.items():
    print(f"{laser} nm {power:>4}: {len(files)} position(s)")

---
## 3 — Beads and their precision

Each acquisition is clustered on its own, then reduced to one row per bead (`_beads.bead_precision`).
`sigma_xy_cm` is a check, not the published quantity (see §7). It also removes the motion each frame
shares with the other beads in the same field.

In [ ]:
tables = []
for (laser, power), files in loc_files.items():
    _, sf_db = mf.cluster_sample(SM_E, files, DATA_ROOT / laser / power, f"{laser}_{power}",
                                 bd.CLUSTER_PARAMS)
    tables.append(bd.bead_precision(sf_db).assign(laser=laser, power=power))
beads = pd.concat(tables, ignore_index=True)
beads.to_csv(FIG_DIR / "bead_precision.csv", index=False)

loc = bd.clean_loc(beads)
colour = bd.clean_colour(bd.colour_long(beads))

summary = beads.groupby(["laser", "power"], sort=False).agg(
    beads=("n", "size"), median_frames=("n", "median"), median_signal=("photons", "median"))
summary["in_loc_plot"] = loc.groupby(["laser", "power"], sort=False).size()
print(f"{len(beads):,} beads seen in >= {bd.MIN_FRAMES} frames; {len(loc):,} in the sigma_xy plot, "
      f"{len(colour):,} bead-pixel pairs in the colour plot")
summary.round(0)

---
## 4 — Figure 2d, as published: all lasers pooled

Top: σ_xy. Bottom: σ_colour, all three pixels (pixel 2 × √2) against the signal on that pixel.
Solid lines: $a/\sqrt{N}+c$ fits (`curve_fit`, `sigma = 1/N`, as published). Dashed lines: 25th–75th
percentiles of the photons per localisation of fluorescent proteins (black) and organic dyes (green).

In [ ]:
N_GRID = np.linspace(100, bd.MAX_PHOTONS, 500)


def fit_table(groups):
    """groups: {name: (N, sigma)} -> a, c with 95 % bootstrap intervals, and sigma at 1e3 / 5e3."""
    rows = {}
    for name, (x, y) in groups.items():
        p, ci = bd.fit_power_law(x, y, n_boot=N_BOOT)
        rows[name] = {"n_beads": len(x), "a": p[0], "a_lo": ci[0, 0], "a_hi": ci[0, 1],
                      "c": p[1], "c_lo": ci[1, 0], "c_hi": ci[1, 1],
                      "at_1e3": bd.power_law(1e3, *p), "at_5e3": bd.power_law(5e3, *p)}
    return pd.DataFrame(rows).T


def range_lines(ax):
    for values, c in [(bd.FLUORESCENT_PROTEINS, "black"), (bd.ORGANIC_DYES, "green")]:
        for v in bd.quartiles(values):
            ax.axvline(v, lw=1, color=c, ls="--")


pooled_fits = fit_table({"sigma_xy / nm": (loc["photons"], loc["sigma_xy"]),
                         "sigma_colour / %": (colour["photons_px"], colour["prec_pct"])})

fig, (ax1, ax2) = plotter.one_column_plot(npanels=2, ratios=[1, 1], height=2.46, width=2.6779528)
fig.tight_layout()
fig.subplots_adjust(hspace=0)
ax1.scatter(loc["photons"], loc["sigma_xy"], s=50, lw=0, color="k", alpha=0.03, marker=".", rasterized=True)
ax1.plot(N_GRID, bd.power_law(N_GRID, *pooled_fits.loc["sigma_xy / nm", ["a", "c"]]), color="k", lw=1.5)
ax2.scatter(colour["photons_px"], colour["prec_pct"], s=50, lw=0, color="r", alpha=0.01, marker=".", rasterized=True)
ax2.plot(N_GRID, bd.power_law(N_GRID, *pooled_fits.loc["sigma_colour / %", ["a", "c"]]), color="darkred", lw=1.5)
for ax in (ax1, ax2):
    range_lines(ax)
    ax.set_xlim(100, bd.MAX_PHOTONS)
    ax.set_ylim(0, 22)
ax1.set_xticklabels([])
ax1.set_ylabel(r"$\sigma_{xy}$/nm")
ax2.set_ylabel(r"$\sigma_{colour}$/%")
ax2.set_xlabel(r"N / detected photoelectrons per frame")
plt.savefig(FIG_DIR / "Precision_Beads_all_excitation_scaledGreen_powerLawFitsLinear.svg", dpi=600, format="svg")
plt.show()

print("Photon ranges (25th-75th percentile): fluorescent proteins "
      f"{bd.quartiles(bd.FLUORESCENT_PROTEINS)}, organic dyes {bd.quartiles(bd.ORGANIC_DYES)}")
pooled_fits.astype(float).round(2)

---
## 5 — The same, split by excitation laser

One column per laser. Top: σ_xy with that laser's fit, plus the pooled fit (grey, dashed). Bottom:
σ_colour, coloured by pixel (blue = pixel 1, green = pixel 2 × √2, red = pixel 3), with that laser's
fit over all three pixels. The colour of the beads differs with the laser, because each laser excites
a different TetraSpeck dye. The weakest pixel (pixel 1 under 561 and 638 nm, a few % of the signal)
sits at low N on the colour plot and carries most of its spread.

In [ ]:
laser_loc_fits = fit_table({l: (d["photons"], d["sigma_xy"]) for l, d in loc.groupby("laser")})
laser_colour_fits = fit_table({l: (d["photons_px"], d["prec_pct"]) for l, d in colour.groupby("laser")})

alpha = 0.3 if FAST_MODE else 0.05
fig, axs = plotter.two_column_plot(nrows=2, ncols=3, height=4.0)
for j, laser in enumerate(bd.LASERS):
    a1, a2 = axs[0, j], axs[1, j]
    d = loc[loc["laser"] == laser]
    a1.scatter(d["photons"], d["sigma_xy"], s=20, lw=0, color=bd.LASER_COLOURS[laser], alpha=alpha,
               marker=".", rasterized=True)
    a1.plot(N_GRID, bd.power_law(N_GRID, *laser_loc_fits.loc[laser, ["a", "c"]]), color="k", lw=1.2)
    a1.plot(N_GRID, bd.power_law(N_GRID, *pooled_fits.loc["sigma_xy / nm", ["a", "c"]]),
            color="grey", lw=1, ls="--")
    a1.set_title(f"{laser} nm excitation ({len(d)} beads)")
    d = colour[colour["laser"] == laser]
    for ch, c in bd.CHANNEL_COLOURS.items():
        e = d[d["channel"] == ch]
        a2.scatter(e["photons_px"], e["prec_pct"], s=20, lw=0, color=c, alpha=alpha, marker=".", rasterized=True)
    a2.plot(N_GRID, bd.power_law(N_GRID, *laser_colour_fits.loc[laser, ["a", "c"]]), color="k", lw=1.2)
    a2.plot(N_GRID, bd.power_law(N_GRID, *pooled_fits.loc["sigma_colour / %", ["a", "c"]]),
            color="grey", lw=1, ls="--")
    a2.set_xlabel("N / photoelectrons per frame")
    for ax in (a1, a2):
        ax.set_xlim(100, bd.MAX_PHOTONS)
        ax.set_ylim(0, 22)
axs[0, 0].set_ylabel(r"$\sigma_{xy}$/nm")
axs[1, 0].set_ylabel(r"$\sigma_{colour}$/%")
plt.savefig(FIG_DIR / "Precision_Beads_by_excitation.svg", dpi=600, format="svg", bbox_inches="tight")
plt.show()

print("sigma_xy fits (nm), a / sqrt(N) + c, 95 % bootstrap intervals over beads:")
display(laser_loc_fits.astype(float).round(2))
print("sigma_colour fits (%), all three pixels:")
display(laser_colour_fits.astype(float).round(2))

---
## 6 — Do the lasers agree where their signal ranges overlap?

Each laser's beads are binned on N (log-spaced bins). For each bin with ≥ 5 beads, the plot shows the
median and its 95 % bootstrap interval. Lasers agree where their intervals overlap. Colour precision
is binned per pixel, on the signal on that pixel. The table lists the bins where at least two lasers
have ≥ 5 beads.

In [ ]:
EDGES = np.geomspace(300, bd.MAX_PHOTONS, 13)
MIN_PER_BIN = 3 if FAST_MODE else 5


def log_x(ax):
    ax.set_xscale("log")
    ax.set_xlim(EDGES[0], EDGES[-1])
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.xaxis.set_minor_formatter(mticker.NullFormatter())


fig, axs = plotter.two_column_plot(nrows=1, ncols=4, height=2.2)
binned = {}
for laser in bd.LASERS:
    d = loc[loc["laser"] == laser]
    binned["sigma_xy", laser] = bd.binned_medians(d["photons"], d["sigma_xy"], EDGES, MIN_PER_BIN, N_BOOT)
    for ch in bd.CHANNELS:
        e = colour[(colour["laser"] == laser) & (colour["channel"] == ch)]
        binned[f"colour_{ch}", laser] = bd.binned_medians(e["photons_px"], e["prec_pct"], EDGES, MIN_PER_BIN, N_BOOT)

panels = [("sigma_xy", r"$\sigma_{xy}$/nm"), ("colour_B", r"pixel 1 $\sigma_{colour}$/%"),
          ("colour_G", r"pixel 2 $\sigma_{colour}$/% (×√2)"), ("colour_R", r"pixel 3 $\sigma_{colour}$/%")]
for ax, (key, ylabel) in zip(axs, panels):
    for laser in bd.LASERS:
        b = binned[key, laser]
        ax.errorbar(b["photons"], b["median"], yerr=[b["median"] - b["lo"], b["hi"] - b["median"]],
                    color=bd.LASER_COLOURS[laser], marker="o", ms=3, lw=1, capsize=2, label=f"{laser} nm")
    log_x(ax)
    ax.set_ylim(bottom=0)
    ax.set_xlabel("N / photoelectrons per frame")
    ax.set_ylabel(ylabel)
axs[0].legend(fontsize=6)
plt.savefig(FIG_DIR / "Precision_Beads_by_excitation_binned.svg", dpi=600, format="svg", bbox_inches="tight")
plt.show()

rows = []
for key, _ in panels:
    wide = pd.concat({l: binned[key, l].set_index("photons")[["n", "median", "lo", "hi"]] for l in bd.LASERS}, axis=1)
    shared = wide.xs("median", axis=1, level=1).notna().sum(axis=1) >= 2
    rows.append(wide[shared].assign(quantity=key))
overlap = pd.concat(rows).reset_index().set_index(["quantity", "photons"])
overlap.to_csv(FIG_DIR / "overlap_binned_medians.csv")
overlap.round(2)

---
## 7 — Check: residual motion common to all beads in a field

AIM removes most of the drift. Some motion shared by all beads in a field may remain, from frame to
frame or as a slow wander. It would add to every bead's σ_xy and does not come from the fit.
`sigma_xy_cm` subtracts, from each frame, the mean offset of the other well-localised beads
(σ_xy 0.01–20 nm) in the same field. Where σ_xy_cm is well below σ_xy, part of the published σ_xy
comes from the sample or stage rather than the fit. This does not change σ_colour.

In [ ]:
cm = beads.dropna(subset=["sigma_xy_cm"])
cm = cm[cm["sigma_xy"].between(*bd.LOC_RANGE_NM, inclusive="neither") & (cm["photons"] < bd.MAX_PHOTONS)]

fig, axs = plotter.two_column_plot(nrows=1, ncols=3, height=2.2)
cm_rows = {}
for ax, laser in zip(axs, bd.LASERS):
    d = cm[cm["laser"] == laser]
    for col, c, label in [("sigma_xy", bd.LASER_COLOURS[laser], "published"),
                          ("sigma_xy_cm", "k", "common motion removed")]:
        b = bd.binned_medians(d["photons"], d[col], EDGES, MIN_PER_BIN, N_BOOT)
        ax.errorbar(b["photons"], b["median"], yerr=[b["median"] - b["lo"], b["hi"] - b["median"]],
                    color=c, marker="o", ms=3, lw=1, capsize=2, label=label)
    log_x(ax)
    ax.set_ylim(0, 22)
    ax.set_title(f"{laser} nm ({len(d)} beads)")
    ax.set_xlabel("N / photoelectrons per frame")
    cm_rows[laser] = {"beads": len(d), "median sigma_xy": d["sigma_xy"].median(),
                      "median sigma_xy_cm": d["sigma_xy_cm"].median(),
                      "median ratio": (d["sigma_xy_cm"] / d["sigma_xy"]).median()}
axs[0].set_ylabel(r"$\sigma_{xy}$/nm")
axs[0].legend(fontsize=6)
plt.savefig(FIG_DIR / "Precision_Beads_common_motion.svg", dpi=600, format="svg", bbox_inches="tight")
plt.show()
pd.DataFrame(cm_rows).T.astype(float).round(2)